# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Wanoleo/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
from google.colab import files
uploaded = files.upload()  # select capstone_features.csv

import pandas as pd, numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import roc_auc_score, classification_report

df = pd.read_csv('capstone_features.csv')
feats = ['imp_prev30','visible_queries','rare_share','anon_share','top_query_share','pos_volatility_60d']
model_data = df.dropna(subset=feats)
X, y = model_data[feats], model_data['is_declining']
print(f'{len(model_data):,} rows ready')

Saving capstone_features.csv to capstone_features (1).csv
102,202 rows ready


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [3]:
print("Finding 1: on a random split, the model looked only marginally")
print("better than the base rate — question: real signal, or client memorization?")
print()
print("Finding 2: feature importance showed pos_volatility_60d as strongest —")
print("question: does that hold on clients the model has never seen?")

Finding 1: on a random split, the model looked only marginally
better than the base rate — question: real signal, or client memorization?

Finding 2: feature importance showed pos_volatility_60d as strongest —
question: does that hold on clients the model has never seen?


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [4]:
# BEFORE: random split
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model_r = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr_r, y_tr_r)
print("BEFORE (random split):")
print(classification_report(y_te_r, model_r.predict(X_te_r), digits=3))

# AFTER: grouped split
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=model_data['client_hash_id']))
X_tr_g, X_te_g = X.iloc[train_idx], X.iloc[test_idx]
y_tr_g, y_te_g = y.iloc[train_idx], y.iloc[test_idx]
model_g = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr_g, y_tr_g)
auc_g = roc_auc_score(y_te_g, model_g.predict_proba(X_te_g)[:, 1])
baseline_auc_g = roc_auc_score(y_te_g, model_data.loc[X_te_g.index, 'pos_volatility_60d'])
print("\nAFTER (client-grouped split):")
print(classification_report(y_te_g, model_g.predict(X_te_g), digits=3))
print(f"Baseline AUC: {baseline_auc_g:.3f} | Model AUC: {auc_g:.3f}")

BEFORE (random split):
              precision    recall  f1-score   support

           0      0.627     0.474     0.540      9389
           1      0.732     0.836     0.781     16162

    accuracy                          0.703     25551
   macro avg      0.679     0.655     0.660     25551
weighted avg      0.693     0.703     0.692     25551


AFTER (client-grouped split):
              precision    recall  f1-score   support

           0      0.466     0.668     0.549     16706
           1      0.800     0.635     0.708     34995

    accuracy                          0.646     51701
   macro avg      0.633     0.651     0.629     51701
weighted avg      0.692     0.646     0.657     51701

Baseline AUC: 0.623 | Model AUC: 0.705


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [5]:
print("Features come from the prior 30/60-day window; label comes from the")
print("following last-30-day window — no overlap.")
print("client_hash_id / content_hash_id used only for joining and for the")
print("GroupShuffleSplit grouping — never passed to the model as features:")
print(feats)

Features come from the prior 30/60-day window; label comes from the
following last-30-day window — no overlap.
client_hash_id / content_hash_id used only for joining and for the
GroupShuffleSplit grouping — never passed to the model as features:
['imp_prev30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share', 'pos_volatility_60d']


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [6]:
print("BEFORE: 'The model predicts which pages will decline.'")
print()
print("AFTER: 'The model ranks pages by observed decline risk, using prior")
print("impressions, query concentration, and position volatility patterns —")
print("validated on clients it never trained on. It does not establish")
print("causation and is not a guarantee for any individual page.'")

BEFORE: 'The model predicts which pages will decline.'

AFTER: 'The model ranks pages by observed decline risk, using prior
impressions, query concentration, and position volatility patterns —
validated on clients it never trained on. It does not establish
causation and is not a guarantee for any individual page.'


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.